# FlowCyt population quantification

This notebook runs the fast integration analogue of the complete research workflow on the committed real-data fixture. The fixture exercises the same patient split, score construction, frozen gates, downstream likelihood, baselines, and uncertainty code. The illustrated all-patient result and its exact reproduction commands live in `docs/usecases/cellpopulation.md`.

## Representation contract

The application owns marker preprocessing and a patient-cross-fitted classifier. A nested reference-patient audit selects the posterior calibration and priors before their density ratios produce five simplex score directions. `fisherbin.mixture_scores_from_posteriors` owns that generic algebra; `fisherbin.fit_scores` owns only the score-to-hard-bin step. An independent reference subset estimates `P(bin | population)`, and the test likelihood consumes counts after all continuous markers and labels have been discarded.

In [ ]:
from pathlib import Path

from examples.cell_population import load_fixture
from examples.cell_population.experiment import run_experiment
from examples.cell_population.figures import make_figure

In [ ]:
data = load_fixture(Path("examples/data/flowcyt_fixture.npz"))
data.features.shape, sorted(set(data.patients))

In [ ]:
result = run_experiment(
    data,
    bin_counts=(5,),
    operating_n_bins=5,
    uncertainty_n_bins=5,
    quick=True,
)
result.metrics["soft_voronoi:5"]

In [ ]:
make_figure(result)

## Full research result

The frozen study range-reads 20,000 composition-preserving events for each of all 30 patients (600,000 cells), trains on 20 reference patients, and evaluates ten untouched patients. Eight learned gates retain 0.982 held-out D-efficiency and reach 0.00196 macro RMSE on the five target fractions. The selected unbinned classifier-ratio baseline reaches 0.00173 macro RMSE. The predeclared random-partition and marker-k-means gates pass at all six tested bin counts.

Generate the bounded sample with `uv run python -m examples.cell_population --download-sample flowcyt-results/flowcyt_sample_20000.npz --max-per-patient 20000 --sample-blocks 16`, then reproduce the figures with `--fixture flowcyt-results/flowcyt_sample_20000.npz --full`.